# Collisions & Consequences
## Analysing the Frequency and Location of Traffic Crashes in Washington, D.C.

**Todi Odedele**  
MATH 014 – Introduction to Data Science  
Howard University | Spring 2026  
Dataset: [Crashes in DC](https://opendata.dc.gov/), DC Open Data Portal / Metropolitan Police Department

## 1. Introduction

Washington, D.C. is the highest city when ranked by population density. The **Crashes in DC** dataset, published by the Metropolitan Police Department through the DC Open Data Portal, has a record of all reported traffic incidents in the District with specifications in location coordinates, crash severity, and casualty counts, date/time of occurrence, and road conditions.

This dataset is incredibly detailed and analytically what some would call a gold mine. It is so useful in geospatial terms, allowing us to see spatial clustering which can be used for analysis of high risk areas. Due to the fact that the data spans over multiple years it allows for seasonal trend analysis. It contains severity and casualty data, allowing us to distinguish minor accidents from fatal ones. It also has public policy relevance. These reports can inform road design, traffic enforcement, and urban planning policy in the nation's capital.

As data scientists, we have an obligation to apply our skills to problems that matter. Traffic safety is a public health issue, and this analysis aims to extract actionable intelligence from raw crash records through rigorous exploratory data analysis and purposeful visualization.

## 2. Research Questions & Objectives

My goal in this analysis is to understand **when, where, and under what conditions** traffic crashes in Washington, D.C. happen and the severity of them. This will help me identify patterns that could reduce crash rates and fatalities. My three research questions that will fuel my analysis are:

**RQ1 — Time frame**  
Do crash frequencies exhibit patterns by hour, day, or season? Are serious crashes distributed differently?

**RQ2 — Location**  
Are there geographic hotspots where crashes concentrate? Does severity vary by location?

**RQ3 — Environmental**  
How do road surface and lighting conditions influence crash severity? Which factor combinations are most dangerous?

## 3. Data Understanding & Cleaning

The dataset was loaded from the DC Open Data Portal, which has crash records reported by the Metropolitan Police Department spanning multiple years.

### 3.1 Cleaning Pipeline Overview

The dataset was loaded from the DC Open Data Portal, which has crash records reported by the Metropolitan Police Department spanning multiple years.

These were my seven cleaning steps applied in order:

- **Column name standardization:** All column names were converted to lowercase with underscores for consistent programmatic access.
- **Duplicate removal:** Duplicate crash records were identified and dropped to avoid any inaccurate effects on my data.
- **Datetime parsing:** The crash date column was parsed to datetime format, and temporal features — year, month, day of week and hour specifically were extracted.
- **Missing value resolution:** Numeric columns were filled with 0 and categorical columns were filled with "Unknown".
- **String standardization:** Whitespace stripped and title casing were applied to all categorical columns.
- **Column pruning:** Columns with less than 70% missing or purely administrative IDs were dropped entirely.
- **Severity labeling:** I made a four-tier severity label that was engineered from fatality, major injury, and minor injury count columns.

### 3.2 Data Cleaning Code

The following code illustrates the datetime parsing and severity labeling steps, the two most analytically connected changes in the cleaning process:

In [ ]:
# ── Library Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# Global style settings
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries loaded successfully.")

In [ ]:

df = pd.read_csv('Crashes_in_DC.csv')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
#Standardize column names 
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('[^a-z0-9_]', '', regex=True)
print("Standardized columns:", df.columns.tolist())

In [ ]:
# Remove duplicate records 
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicates removed: {dupes_before} | Rows remaining: {len(df):,}")

In [ ]:

date_col = [c for c in df.columns if 'date' in c or 'time' in c][0]
print(f"Parsing date column: '{date_col}'")

df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

df['year']        = df[date_col].dt.year
df['month']       = df[date_col].dt.month
df['month_name']  = df[date_col].dt.strftime('%b')
df['day_of_week'] = df[date_col].dt.day_name()
df['hour']        = df[date_col].dt.hour
df['season']      = df['month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring',  4:'Spring', 5:'Spring',
    6:'Summer',  7:'Summer', 8:'Summer',
    9:'Fall',   10:'Fall',  11:'Fall'
})

print(f"Date range: {df[date_col].min().date()} to {df[date_col].max().date()}")

In [ ]:
#Resolve missing values
numeric_fill = [c for c in df.select_dtypes(include='number').columns if df[c].isnull().sum() > 0]
df[numeric_fill] = df[numeric_fill].fillna(0)

cat_fill = [c for c in df.select_dtypes(include='object').columns if df[c].isnull().sum() > 0]
df[cat_fill] = df[cat_fill].fillna('Unknown')

print(f"Remaining nulls after fill: {df.isnull().sum().sum()}")

In [ ]:
# Standardize string entries 
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.title()
print("String standardization complete.")

In [ ]:
#Drop irrelevant or redundant columns 
drop_candidates = [c for c in df.columns
                   if (df[c].isnull().sum() / len(df) > 0.70)
                   or c in ['objectid', 'globalid', 'shape']]
df = df.drop(columns=drop_candidates, errors='ignore')
print(f"Dropped columns: {drop_candidates}")
print(f"Final shape: {df.shape}")

In [ ]:
# Create severity label
fatal_col  = [c for c in df.columns if 'fatal' in c]
major_col  = [c for c in df.columns if 'majorinjur' in c]
minor_col  = [c for c in df.columns if 'minorinjur' in c]

if fatal_col and major_col:
    df['severity'] = 'Property Damage Only'
    if minor_col:
        df.loc[df[minor_col[0]] > 0, 'severity'] = 'Minor Injury'
    df.loc[df[major_col[0]] > 0, 'severity'] = 'Major Injury'
    df.loc[df[fatal_col[0]] > 0, 'severity'] = 'Fatal'
    print("Severity column created:")
    print(df['severity'].value_counts())
else:
    print("Severity columns not found: check column names.")

## 4. Visualizations & Interpretation

### Bar graph, Crash Frequency by Hour of Day

**Intent:** This histogram examines the distribution of crashes across the 24 hour clock. By identifying peak and trough hours, it directly addresses Research Question 1, whether crashes follow temporal patterns. Rush-hour peaks support targeted enforcement and infrastructure investment strategies.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

hourly_counts = df['hour'].value_counts().sort_index()

bars = ax.bar(hourly_counts.index, hourly_counts.values,
              color=sns.color_palette('muted')[0], edgecolor='white', linewidth=0.5)


peak = hourly_counts.idxmax()
ax.bar(peak, hourly_counts[peak], color='crimson', edgecolor='white', label=f'Peak: {peak}:00')
ax.annotate(f'Peak
{hourly_counts[peak]:,}',
            xy=(peak, hourly_counts[peak]),
            xytext=(peak + 1.5, hourly_counts[peak] * 0.95),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, color='crimson', fontweight='bold')

ax.set_xlabel('Hour of Day (0 = Midnight)', fontsize=12)
ax.set_ylabel('Number of Crashes', fontsize=12)
ax.set_title('Traffic Crash Frequency by Hour of Day in Washington, D.C.', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
sns.despine()
plt.tight_layout()
plt.savefig('plot1_hourly.png', dpi=150, bbox_inches='tight')
plt.show()

*Figure 4.1 — Bar chart of crash frequency by hour of day. Red bar marks the peak hour.*

This histogram observed the distribution of crashes in total throughout a 24-hour period by hour. By identifying the peak and bases we can see a clear solution and answer to my first research question on whether crashes follow temporal patterns. Peak rush hour time supports targeted enforcement and infrastructure investment strategies would be beneficial to mitigate the amount of crashes.

### Donut Chart, Crash Severity Distribution

**Intent:** A donut chart gives an understanding of the proportion of crashes by severity tier. This is foundational context for all severity-related downstream analyses and directly frames the stakes of the problem.

In [ ]:
severity_order  = ['Property Damage Only', 'Minor Injury', 'Major Injury', 'Fatal']
severity_colors = ['#4e9af1', '#f6c90e', '#f4845f', '#d62828']

sev_data = df['severity'].value_counts().reindex(severity_order).fillna(0)

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    sev_data,
    labels=sev_data.index,
    autopct='%1.1f%%',
    colors=severity_colors,
    startangle=140,
    pctdistance=0.82,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight('bold')

ax.set_title('Distribution of Crash Severity in Washington, D.C.', fontsize=14, fontweight='bold', pad=20)
ax.legend(wedges, [f"{k}: {int(v):,}" for k, v in sev_data.items()],
          title='Severity Level', loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('plot2_severity_pie.png', dpi=150, bbox_inches='tight')
plt.show()

*Figure 4.3 — Donut chart: crash severity proportions. Fatal and Major Injury crashes represent a critical minority.*

**Interpretation:** Most motor vehicle accidents in DC only result in some type of property damage, expected in an urban environment with lower speeds. However, the share of major injuries and fatalities highlights that even a small percentage of crashes carries severe human cost. Fatal crashes, at 2.5%, represent the most preventable tragedies and warrant the deepest analytical attention.

### Comparative bar chart, Crashes by Day of Week and Severity

This grouped bar chart examines whether crash severity varies by day of week, which relates to my first research question: Weekend patterns which may be linked to alcohol consumption have different policy implications than weekday patterns.

In [ ]:
dow_order      = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
severity_order = ['Property Damage Only', 'Minor Injury', 'Major Injury', 'Fatal']

dow_sev = (df.groupby(['day_of_week', 'severity'])
             .size()
             .reset_index(name='count'))
dow_sev['day_of_week'] = pd.Categorical(dow_sev['day_of_week'], categories=dow_order, ordered=True)
dow_sev = dow_sev.sort_values('day_of_week')

fig, ax = plt.subplots(figsize=(14, 6))
pivot = dow_sev.pivot(index='day_of_week', columns='severity', values='count').fillna(0)
pivot = pivot.reindex(columns=severity_order)
pivot.plot(kind='bar', ax=ax, color=['#4e9af1','#f6c90e','#f4845f','#d62828'],
           edgecolor='white', linewidth=0.4)

ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Number of Crashes', fontsize=12)
ax.set_title('Crash Counts by Day of Week and Severity', fontsize=14, fontweight='bold')
ax.set_xticklabels(dow_order, rotation=30, ha='right')
ax.legend(title='Severity', loc='upper right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
sns.despine()
plt.tight_layout()
plt.savefig('plot3_dow_severity.png', dpi=150, bbox_inches='tight')
plt.show()

*Figure 4.4 — Grouped bar chart: crash counts by day of week, colored by severity tier.*

Weekdays particularly **Friday** have the highest total crash volumes. Saturday and Sunday show a relatively higher proportion of major injury and fatal crashes per incident, consistent with nighttime driving behaviors on weekend nights. Weekday policies should focus on volume reduction whereas weekend interventions should target impaired and high-speed driving.

### Boxplot: Injuries per Crash by Season

**Intent:** This boxplot examines whether crash-injury counts vary by season, directly testing Research Question 1's seasonal component. Snowstorms, icy roads, and reduced visibility in winter could produce different injury distributions than warmer months.

In [ ]:
total_injury_col = [c for c in df.columns if 'total' in c and ('injur' in c or 'person' in c)]

if not total_injury_col:
    injury_cols = [c for c in df.columns if 'injur' in c and df[c].dtype in ['float64','int64']]
    if injury_cols:
        df['total_injuries'] = df[injury_cols].sum(axis=1)
        total_injury_col = ['total_injuries']

if total_injury_col:
    col = total_injury_col[0]
    season_order = ['Winter', 'Spring', 'Summer', 'Fall']
    palette = {'Winter':'#74b9ff','Spring':'#55efc4','Summer':'#ffeaa7','Fall':'#e17055'}

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=df, x='season', y=col, order=season_order,
                palette=palette, flierprops=dict(marker='o', markersize=3, alpha=0.4),
                ax=ax)
    ax.set_xlabel('Season', fontsize=12)
    ax.set_ylabel('Number of Injuries per Crash', fontsize=12)
    ax.set_title('Injury Count Distribution per Crash by Season', fontsize=14, fontweight='bold')

    for i, season in enumerate(season_order):
        median = df[df['season']==season][col].median()
        ax.text(i, median + 0.05, f'Med={median:.2f}', ha='center', fontsize=9, color='navy')

    sns.despine()
    plt.tight_layout()
    plt.savefig('plot4_season_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No suitable injury column found.")

*Figure 4.5 — Boxplot of injuries per crash by season. Medians annotated above each box.*

**Interpretation:** Median injuries per crash remain relatively consistent across seasons, but the distribution of outliers differs. Winter shows a slightly wider spread of high-injury crashes, potentially related to icy conditions causing multi-vehicle pileups. Summer shows fewer extreme outliers. Season alone is not a strong predictor of injury severity but contributes to the overall risk environment.

### Crash Heatmap: Hour × Day of Week

**Intent:** This heatmap simultaneously encodes two temporal dimensions — hour of day and day of week — to reveal the precise windows when crash risk peaks. This multivariate view is far more informative than examining each dimension independently.

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

heat_data = (df.groupby(['day_of_week','hour'])
               .size()
               .reset_index(name='count')
               .pivot(index='day_of_week', columns='hour', values='count')
               .fillna(0)
               .reindex(dow_order))

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heat_data, cmap='YlOrRd', ax=ax, linewidths=0.3, linecolor='white',
            annot=False, cbar_kws={'label': 'Crash Count'})

ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Day of Week', fontsize=12)
ax.set_title('Crash Frequency Heatmap: Hour of Day × Day of Week', fontsize=14, fontweight='bold')
ax.set_xticklabels(range(0, 24), rotation=0, fontsize=9)
ax.set_yticklabels(dow_order, rotation=0)

plt.tight_layout()
plt.savefig('plot5_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

*Figure 4.7 — Heatmap of crash frequency. Darkest cells indicate highest-risk hour/day combinations.*

The darkest cells cluster in two zones: one is weekday afternoon hours (3–6 PM, Mon–Fri), consistent with after work rush hour times and after-school exit times; and the other is late Friday/Saturday nights (11 PM–2 AM), most likely reflecting nightlife-related crashes. This visual time scale shows the need for highly targeted enforcement scheduling, concentrating resources on specific high-risk windows rather than uniform 24/7 patrol.

### Line Chart, Monthly Crash Trend

**Intent:** A Plotly interactive line chart allows us to explore the month by month trend in crash counts across all years, surfacing longer term temporal trends including any anomalies such as the COVID-related drop in 2020.

In [ ]:
date_col_name = [c for c in df.columns if 'date' in c and df[c].dtype == 'datetime64[ns]'][0]

monthly = (df.groupby(df[date_col_name].dt.to_period('M'))
             .size()
             .reset_index(name='crash_count'))
monthly[date_col_name] = monthly[date_col_name].dt.to_timestamp()

fig = px.line(
    monthly,
    x=date_col_name,
    y='crash_count',
    title='Monthly Crash Counts in Washington, D.C. Over Time',
    labels={date_col_name: 'Month', 'crash_count': 'Number of Crashes'},
    template='plotly_white',
    markers=True
)
fig.update_traces(
    line_color='#d62828',
    marker=dict(size=4),
    hovertemplate='<b>%{x|%b %Y}</b><br>Crashes: %{y:,}<extra></extra>'
)
fig.update_layout(
    font_size=13,
    hovermode='x unified',
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True, tickformat=',')
)
fig.show()

A Plotly interactive line chart explored month-by-month crash counts across all years. The chart revealed seasonal oscillations (higher in warmer months) and a pronounced dip in 2020 consistent with COVID-19 pandemic lockdowns. Post-pandemic years show counts below pre-pandemic peaks, potentially reflecting a combination of safety improvements and persistent remote work patterns.

### Scatter, Crash Map by Severity

**Intent:** A geospatial scatter map plots each crash as a colored point on a real DC base map, directly answering Research Question 2. Hover text surfaces specific incident details.

In [ ]:
lat_col = [c for c in df.columns if 'lat' in c][0]
lon_col = [c for c in df.columns if 'lon' in c or 'lng' in c][0]

map_df = df[
    (df[lat_col].between(38.79, 38.99)) &
    (df[lon_col].between(-77.12, -76.91)) &
    (df['severity'].notna())
].copy()

if len(map_df) > 30000:
    map_df = map_df.sample(30000, random_state=42)

severity_color_map = {
    'Property Damage Only': '#4e9af1',
    'Minor Injury': '#f6c90e',
    'Major Injury': '#f4845f',
    'Fatal': '#d62828'
}

fig = px.scatter_mapbox(
    map_df,
    lat=lat_col, lon=lon_col,
    color='severity',
    color_discrete_map=severity_color_map,
    zoom=11,
    center={'lat': 38.905, 'lon': -77.036},
    mapbox_style='carto-positron',
    title='Geographic Distribution of Crashes in Washington, D.C. by Severity',
    opacity=0.55,
    hover_data={'severity': True, lat_col: False, lon_col: False}
)
fig.update_layout(
    legend_title='Crash Severity',
    font_size=12,
    margin=dict(r=0, t=40, l=0, b=0)
)
fig.show()

A geospatial scatter map plotted each crash as a colored point on a real DC base map, directly answering Research Question 2. High-density clusters appear along **Georgia Avenue**, **Pennsylvania Avenue**, **New York Avenue**, and the downtown core. Fatal crashes concentrate disproportionately on high-speed corridors like South Capitol Street and Benning Road. **Ward 5 and Ward 7** east-of-the-river corridors show notable concentrations of serious crashes — a finding with direct equity implications for DC's transportation policy.

## 5. Additional Analyses

### Pedestrian & Cyclist Crash Trends Over Time

**Motivation:** DC has invested heavily in Vision Zero and pedestrian/cyclist infrastructure. This analysis asks whether those investments are reflected in declining pedestrian and cyclist crash counts over time.

In [ ]:
ped_col  = [c for c in df.columns if 'pedestrian' in c and df[c].dtype in ['float64','int64']]
bike_col = [c for c in df.columns if ('bicycle' in c or 'cyclist' in c) and df[c].dtype in ['float64','int64']]

if ped_col and bike_col:
    yearly = df.groupby('year').agg(
        ped_injured=(ped_col[0], 'sum'),
        bike_injured=(bike_col[0], 'sum')
    ).reset_index()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(yearly['year'], yearly['ped_injured'], 'o-', color='#e84393', label='Pedestrians Injured', lw=2)
    ax.plot(yearly['year'], yearly['bike_injured'], 's-', color='#00b894', label='Cyclists Injured', lw=2)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Annual Pedestrian and Cyclist Injuries in DC (Trend Analysis)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_xticks(yearly['year'].unique())
    ax.tick_params(axis='x', rotation=45)
    sns.despine()
    plt.tight_layout()
    plt.savefig('plotA_ped_bike_trend.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Pedestrian/cyclist columns not found — adjust column name matching.")

*Figure 5.1 — Annual pedestrian and cyclist injuries in DC (2015–2024). Note the 2020 COVID-19 dip.*

**Interpretation:** Both pedestrian and cyclist injury counts trended negatively from 2015–2019 before the 2020 pandemic caused a sharp drop. Post-pandemic recovery years (2021–2024) show counts lower than pre-pandemic peaks, potentially reflecting genuine safety improvements from Vision Zero investments alongside remote work patterns. Sustained monitoring is needed to confirm the trend.

## 6. Conclusions & Limitations

- **RQ1:** Clear bimodal weekday crash peaks (AM/PM commute); late-night weekends show elevated fatal crash proportions. 
- **RQ2:** Crash map reveals dense clustering on major corridors mostly east of the potomac river. Fatal crashes concentrate on high speed roads.
- **RQ3:** Adverse road conditions appear to amplify crash severity however formal statistical tests were beyond scope.

### Key Limitations

- **Reporting bias:** Minor property damage crashes are frequently not reported to police.
- **Missing covariates:** Alcohol involvement, vehicle speed, and driver demographics are incompletely recorded.
- **Observational data:** Correlations cannot be interpreted as causal without controlled study designs.